In [1]:
from ultralytics import YOLO
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

MODEL_PATH = (
    PROJECT_ROOT
    / "results"
    / "augmented_yolo26n"
    / "weights"
    / "best.pt"
)

model = YOLO(str(MODEL_PATH))

print("Final model loaded successfully!")
print("Classes:")
print(model.names)

Final model loaded successfully!
Classes:
{0: 'Baton', 1: 'Pliers', 2: 'Hammer', 3: 'Powerbank', 4: 'Scissors', 5: 'Wrench', 6: 'Gun', 7: 'Bullet', 8: 'Sprayer', 9: 'HandCuffs', 10: 'Knife', 11: 'Lighter'}


In [2]:
TEST_DIR = PROJECT_ROOT / "data" / "images" / "test"

sample_image = next(TEST_DIR.glob("*.jpg"))

print("Testing image:")
print(sample_image)

Testing image:
/Users/priyansh/Downloads/airport-baggage-screening/data/images/test/PID_xray_easy04194.jpg


In [3]:
results = model.predict(
    source=str(sample_image),
    imgsz=640,
    conf=0.25,
    device="mps"
)

results[0].show()


image 1/1 /Users/priyansh/Downloads/airport-baggage-screening/data/images/test/PID_xray_easy04194.jpg: 576x640 1 Sprayer, 7.9ms
Speed: 240.1ms preprocess, 7.9ms inference, 16.9ms postprocess per image at shape (1, 3, 576, 640)


In [4]:
result = results[0]

if result.boxes is not None and len(result.boxes) > 0:

    for box in result.boxes:

        class_id = int(box.cls[0])
        confidence = float(box.conf[0])

        class_name = model.names[class_id]

        print(
            f"Detected: {class_name:10s} "
            f"| Confidence: {confidence:.2%}"
        )

else:
    print("No suspicious objects detected.")

Detected: Sprayer    | Confidence: 89.22%


In [5]:
CONFIDENCE_THRESHOLD = 0.50

detections = []

if result.boxes is not None:

    for box in result.boxes:

        class_id = int(box.cls[0])
        confidence = float(box.conf[0])
        class_name = model.names[class_id]

        if confidence >= CONFIDENCE_THRESHOLD:

            detections.append({
                "object": class_name,
                "confidence": confidence
            })


if len(detections) > 0:

    print("⚠️ SUSPICIOUS BAGGAGE")
    print("-" * 40)

    for detection in detections:
        print(
            f"{detection['object']} "
            f"→ {detection['confidence']:.2%}"
        )

else:

    print("✅ NO SUSPICIOUS OBJECT DETECTED")

⚠️ SUSPICIOUS BAGGAGE
----------------------------------------
Sprayer → 89.22%
